# S6E10 — 05 CatBoost & XGBoost on `X_final`

> **Pipeline position:** 03 features → 04 LightGBM → **05 CatBoost/XGB** → 06 MLP → … → 11 stack

| | |
|---|---|
| **Purpose** | OOF and test predictions of CatBoost and XGBoost on the frozen feature set, same 5 folds and in-fold target encoding as everywhere else. They feed the stack (`11`). |
| **Input** | `data/processed/X_final_*.parquet`, `y_train.parquet` |
| **Output** | `preds/oof_cat_v1|cat_v2|xgb.npy`, `preds/test_*.npy` |
| **Runtime** | CatBoost ≈ 7 min per variant, XGBoost ≈ 1 min (Mac M4, CPU) |

**No Optuna here:** CatBoost tuning gave only +0.0001 before, so compute is better spent on neural networks, which add more *diversity* to the stack.

| Flag | What | Why |
|---|---|---|
| `RUN_CAT_V1` | CatBoost with tuned params, `flight_distance` numeric + in-fold TE | direct counterpart of the earlier tuned CatBoost |
| `RUN_CAT_V2` | v1 + extra categorical copy of `flight_distance` (CatBoost's own ordered target statistics) | the one real experiment: can native categorical handling beat TE? |
| `RUN_XGB` | XGBoost, baseline params | diversity for the stack |

**Always run `SMOKE = True` first (1–2 min):** 40k rows, 1 fold, to check the code works on your machine.

Reference: tuned LightGBM on `X_final` = CV 0.96072 (0.96088 after `04`).

> **Glossary:** *Ordered target statistics* — CatBoost's own leak-free target encoding: each row is encoded using only the rows that precede it in a random permutation. *Symmetric trees* — CatBoost uses the same split at every node of a level, which makes trees fast and strongly regularised.

## 1. Setup and switches
`USE_GPU = False` on a Mac; switch to `True` only on Kaggle with a GPU. The parquet files from `03` are loaded; the assert guards against train/test column mismatch.

In [1]:
import os, time, numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from catboost import CatBoostClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ---------------- switches ----------------
SMOKE      = False    # True: 40k rows, 1 fold (code check). Set False for the real run.
USE_GPU    = False   # False on Mac. True only on Kaggle with GPU enabled.
RUN_CAT_V1 = True
RUN_CAT_V2 = True
RUN_XGB    = True
# ------------------------------------------

N_FOLDS, RANDOM_STATE = 5, 42
CAT_COLS = ['gender', 'customer_type', 'type_of_travel', 'class']

# find the parquet files (local project folder or Kaggle working dir)
CANDIDATES = [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')]
PROC_DIR = next(p for p in CANDIDATES if (p / 'X_final_train.parquet').exists())
PRED_DIR = Path('/kaggle/working/preds') if Path('/kaggle/working').exists() else next((p for p in (Path('preds'), Path('../preds')) if p.exists()), Path('preds'))
PRED_DIR.mkdir(parents=True, exist_ok=True)
print('data from:', PROC_DIR.resolve(), '| predictions to:', PRED_DIR.resolve())

X      = pd.read_parquet(PROC_DIR / 'X_final_train.parquet')
X_test = pd.read_parquet(PROC_DIR / 'X_final_test.parquet')
y      = pd.read_parquet(PROC_DIR / 'y_train.parquet')['y'].astype(int)
assert list(X.columns) == list(X_test.columns) and len(X) == len(y)
print('X:', X.shape, '| X_test:', X_test.shape, '| positive rate:', round(y.mean(), 3))

if SMOKE:
    rng = np.random.RandomState(0)
    idx = rng.choice(len(X), 40000, replace=False)
    X, y = X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    X_test = X_test.iloc[:5000].reset_index(drop=True)
    print('SMOKE MODE: 40k rows, 5k test rows, 1 fold, 100 trees')

# Same 5 folds as the whole project (seed 42).
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(skf.split(X, y))
if SMOKE: FOLDS = FOLDS[:1]
MAX_TREES = 100 if SMOKE else 1000

data from: /Users/deonissx/Documents/Hackathons/kaggle-competitions/predicting_airline_satisfaction_s6e10/data/processed | predictions to: /Users/deonissx/Documents/Hackathons/kaggle-competitions/predicting_airline_satisfaction_s6e10/preds
X: (699635, 45) | X_test: (299844, 45) | positive rate: 0.444


## 2. Helpers
`add_te_flight_distance` is the in-fold target encoding (same as in all earlier notebooks).
`run_cv` trains one model per fold, stores OOF + averaged test predictions, prints per-fold AUC and time.

In [2]:
# In-fold target encoding of the route id: the encoder is fitted on the fold's training part only (inner cv=5), then applied to validation and test. Same helper as in 03/04.
def add_te_flight_distance(X_fit, y_fit, other_frames, smooth='auto'):
    enc = TargetEncoder(smooth=smooth, cv=5, shuffle=True, random_state=RANDOM_STATE)
    key = lambda d: d[['flight_distance']].astype(str)
    X_fit = X_fit.copy()
    X_fit['te_flight_distance'] = enc.fit_transform(key(X_fit), y_fit)[:, 0]
    out = []
    for f in other_frames:
        f = f.copy(); f['te_flight_distance'] = enc.transform(key(f))[:, 0]; out.append(f)
    return X_fit, out

# Generic 5-fold loop shared by all models: target-encode -> model-specific prep -> fit -> store OOF + averaged test prediction.
def run_cv(name, fit_predict, prep=lambda d: d):
    """fit_predict(X_tr, y_tr, X_val, y_val, X_te) -> (val_pred, test_pred)"""
    oof = np.full(len(X), np.nan); test_pred = np.zeros(len(X_test)); aucs = []
    t0 = time.time()
    for k, (tr, va) in enumerate(FOLDS, 1):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y.iloc[tr], y.iloc[va]
        # TE must be recomputed inside every fold; doing it once on all training rows would leak labels into validation.
        X_tr, (X_va, X_te) = add_te_flight_distance(X_tr, y_tr, [X_va, X_test])
        X_tr, X_va, X_te = prep(X_tr), prep(X_va), prep(X_te)
        pv, pt = fit_predict(X_tr, y_tr, X_va, y_va, X_te)
        oof[va] = pv; test_pred += pt / len(FOLDS)
        aucs.append(roc_auc_score(y_va, pv))
        print(f'  [{name}] fold {k}/{len(FOLDS)} AUC {aucs[-1]:.5f} | elapsed {(time.time()-t0)/60:.1f} min', flush=True)
    mask = ~np.isnan(oof)
    cv = roc_auc_score(y[mask], oof[mask])
    print(f'==> {name}: CV AUC {cv:.5f} | folds mean {np.mean(aucs):.5f} +- {np.std(aucs):.5f}')
    if not SMOKE:
        np.save(PRED_DIR / f'oof_{name}.npy', oof); np.save(PRED_DIR / f'test_{name}.npy', test_pred)
        print(f'    saved preds/oof_{name}.npy, preds/test_{name}.npy')
    return oof, test_pred, cv

## 3. CatBoost
Tuned parameters come from the earlier Optuna run (3-fold, 30% of data, on the OLD features): depth 6, lr 0.0434, l2 0.0245, bagging_temperature 0.436, random_strength 0.834.

Two deliberate differences from the Kaggle-GPU version:
- `bootstrap_type='Bayesian'` is set explicitly: `bagging_temperature` only works with Bayesian bootstrap (GPU used it by default, CPU does not).
- `min_data_in_leaf=13` is dropped: it is only valid for `Lossguide`/`Depthwise` trees, while we use the default symmetric trees (it had no effect before).

CatBoost needs categoricals as strings, so they are cast with `astype(str)`.

In [3]:
# Parameters from an earlier Optuna run; see the markdown above for the two CPU-specific adjustments.
CAT_PARAMS = dict(
    iterations=MAX_TREES, loss_function='Logloss', eval_metric='AUC',
    early_stopping_rounds=50, random_seed=RANDOM_STATE, verbose=0,
    learning_rate=0.04336429500029207, depth=6, l2_leaf_reg=0.024485951255744317,
    bootstrap_type='Bayesian', bagging_temperature=0.43557390422246217,
    random_strength=0.8336105913113857,
)
if USE_GPU: CAT_PARAMS['task_type'] = 'GPU'
else:       CAT_PARAMS['thread_count'] = -1

def prep_cat_v1(d):
    d = d.copy()
    for c in CAT_COLS: d[c] = d[c].astype(str)
    return d

# v2 = v1 + an extra categorical copy of the route id, so CatBoost applies its own ordered target statistics on top of our TE.
def prep_cat_v2(d):
    d = prep_cat_v1(d)
    d['flight_distance_cat'] = d['flight_distance'].astype(str)   # extra categorical copy of the route id
    return d

def make_cat_fit(cat_features):
    def fit_predict(X_tr, y_tr, X_va, y_va, X_te):
        m = CatBoostClassifier(**CAT_PARAMS, cat_features=cat_features)
        m.fit(X_tr, y_tr, eval_set=(X_va, y_va))
        return m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]
    return fit_predict

res = {}
if RUN_CAT_V1:
    res['cat_v1'] = run_cv('cat_v1', make_cat_fit(CAT_COLS), prep_cat_v1)
if RUN_CAT_V2:
    res['cat_v2'] = run_cv('cat_v2', make_cat_fit(CAT_COLS + ['flight_distance_cat']), prep_cat_v2)

  [cat_v1] fold 1/5 AUC 0.96074 | elapsed 1.3 min
  [cat_v1] fold 2/5 AUC 0.95971 | elapsed 2.7 min
  [cat_v1] fold 3/5 AUC 0.96140 | elapsed 4.1 min
  [cat_v1] fold 4/5 AUC 0.96074 | elapsed 5.3 min
  [cat_v1] fold 5/5 AUC 0.96060 | elapsed 6.8 min
==> cat_v1: CV AUC 0.96064 | folds mean 0.96064 +- 0.00054
    saved preds/oof_cat_v1.npy, preds/test_cat_v1.npy
  [cat_v2] fold 1/5 AUC 0.96060 | elapsed 1.3 min
  [cat_v2] fold 2/5 AUC 0.95965 | elapsed 2.8 min
  [cat_v2] fold 3/5 AUC 0.96138 | elapsed 4.3 min
  [cat_v2] fold 4/5 AUC 0.96078 | elapsed 5.9 min
  [cat_v2] fold 5/5 AUC 0.96075 | elapsed 7.5 min
==> cat_v2: CV AUC 0.96062 | folds mean 0.96063 +- 0.00056
    saved preds/oof_cat_v2.npy, preds/test_cat_v2.npy


## 4. XGBoost
Baseline parameters (the ones that gave CV 0.95995 on the old features). Categoricals as pandas `category`, `enable_categorical=True`.

In [4]:
# Untuned, sensible defaults: the goal is a different tree-building style (level-wise growth, different regularisation) for the stack, not a record score.
XGB_PARAMS = dict(
    n_estimators=MAX_TREES, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', enable_categorical=True, eval_metric='auc', early_stopping_rounds=50,
    random_state=RANDOM_STATE, device='cuda' if USE_GPU else 'cpu', n_jobs=-1,
)

def prep_xgb(d):
    d = d.copy()
    for c in CAT_COLS: d[c] = d[c].astype('category')
    return d

def xgb_fit_predict(X_tr, y_tr, X_va, y_va, X_te):
    m = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=0)
    return m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]

if RUN_XGB:
    res['xgb'] = run_cv('xgb', xgb_fit_predict, prep_xgb)

  [xgb] fold 1/5 AUC 0.96074 | elapsed 0.2 min
  [xgb] fold 2/5 AUC 0.95981 | elapsed 0.4 min
  [xgb] fold 3/5 AUC 0.96136 | elapsed 0.5 min
  [xgb] fold 4/5 AUC 0.96080 | elapsed 0.7 min
  [xgb] fold 5/5 AUC 0.96061 | elapsed 0.9 min
==> xgb: CV AUC 0.96066 | folds mean 0.96067 +- 0.00050
    saved preds/oof_xgb.npy, preds/test_xgb.npy


## 5. Summary and comparison with LightGBM
Prints a comparison table. The optional correlation check looks for `preds/oof_lgbm_prof_orig.npy` (an old file name, from the first LightGBM run); with the current `oof_lgbm_tuned_final.npy` the same correlations are shown in `11_stack`, so it is fine that this cell skips it.

In [5]:
print('{:8s} {:>9s}'.format('model', 'CV AUC'))
for k, (_, _, cv) in res.items(): print('{:8s} {:9.5f}'.format(k, cv))
print('reference: LGBM tuned on X_final 0.96072')

lgbm_path = PRED_DIR / 'oof_lgbm_prof_orig.npy'
if lgbm_path.exists() and not SMOKE:
    o_l = np.load(lgbm_path)
    for k, (oof, _, _) in res.items():
        print(f'{k}: corr with LGBM {np.corrcoef(o_l, oof)[0,1]:.4f} | mean(LGBM, {k}) AUC {roc_auc_score(y, (o_l+oof)/2):.5f}')
elif not SMOKE:
    print('(put oof_lgbm_prof_orig.npy into', PRED_DIR, 'to see correlation with LGBM)')
if SMOKE: print('SMOKE finished OK -> set SMOKE = False and run again for the real thing.')

model       CV AUC
cat_v1     0.96064
cat_v2     0.96062
xgb        0.96066
reference: LGBM tuned on X_final 0.96072
(put oof_lgbm_prof_orig.npy into preds to see correlation with LGBM)


## Results & decisions

| Model | CV AUC | Time (CPU) | Notes |
|---|---|---|---|
| `cat_v1` | **0.96064** | 6.8 min | folds 0.96074 / 0.95971 / 0.96140 / 0.96074 / 0.96060 |
| `cat_v2` | 0.96062 | 7.5 min | extra categorical copy of the route id |
| `xgb` | **0.96066** | 0.9 min | untuned baseline parameters |
| *LightGBM tuned (reference)* | *0.96088* | | notebook `04` |

- **Native CatBoost categorical handling did not beat target encoding:** `cat_v2` (0.96062) ≈ `cat_v1` (0.96064). The difference is far below the fold-to-fold noise (±0.0005), so the simpler v1 is kept as the main CatBoost; v2 stays in the stack only as a slightly different view.
- **XGBoost is as good as CatBoost at one-seventh of the time**, even untuned. Tuning would not change the picture: all three GBDTs sit at 0.9606–0.9609.
- **Takeaway:** the GBDT family has hit a plateau (~0.9607–0.9609) on these features. Further gains must come from *different kinds* of models (neural networks, TabPFN) — that is why the project continues with `06` (MLP) and `07` (RealMLP).